<a href="https://colab.research.google.com/github/ahmadrafi2026/Pemerograman-Dasar/blob/main/Tugas_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Tugas

# Nama : Ahmad Rafi'i Mailkul Rauf
# NIM  : 4232501036

In [ ]:
def heat_rate(batubara_t, energi_mwh, nilai_kalor_kcal_kg=4400):
    """Menghitung heat rate (kcal/kWh) dari batubara dan energi."""
    if energi_mwh <= 0:
        raise ValueError("energi_mwh harus lebih dari 0")
    return batubara_t * nilai_kalor_kcal_kg / energi_mwh

def efisiensi_termal(heat_rate_kcal_kwh):
    """Menghitung efisiensi termal (%) berdasarkan heat rate."""
    if heat_rate_kcal_kwh <= 0:
        raise ValueError("heat_rate harus lebih dari 0")
    return 860 / heat_rate_kcal_kwh * 100

def capacity_factor(energi_mwh, daya_mw=330, jam=24):
    """Menghitung capacity factor (%) selama periode operasi."""
    if energi_mwh <= 0:
        raise ValueError("energi_mwh harus lebih dari 0")
    if daya_mw <= 0 or jam <= 0:
        raise ValueError("daya_mw dan jam harus lebih dari 0")
    return energi_mwh / (daya_mw * jam) * 100

def konsumsi_spesifik(batubara_t, energi_mwh):
    """Menghitung konsumsi batubara spesifik (t/MWh)."""
    if energi_mwh <= 0:
        raise ValueError("energi_mwh harus lebih dari 0")
    return batubara_t / energi_mwh

def status_hari(efisiensi, jam_operasi):
    """Menentukan status: TIDAK PENUH, EFISIENSI RENDAH, atau NORMAL."""
    if jam_operasi < 24:
        return "TIDAK PENUH"
    if efisiensi < 36.3:
        return "EFISIENSI RENDAH"
    return "NORMAL"

#Uji assert berdasarkan hitungan manual
assert heat_rate(1, 2) == 2200                       # 1 x 4400 / 2
assert round(efisiensi_termal(2200), 2) == 39.09    # 860 / 2200 x 100
assert capacity_factor(7920, 330, 24) == 100       # 7920 / (330 x 24) x 100
assert konsumsi_spesifik(100, 200) == 0.5          # 100 / 200
assert status_hari(35.0, 18) == "TIDAK PENUH"
assert status_hari(36.0, 24) == "EFISIENSI RENDAH"
assert status_hari(37.0, 24) == "NORMAL"

#Uji penanganan energi nol/negatif
for fungsi, args in [
    (heat_rate, (1, 0)),
    (capacity_factor, (0, 330, 24)),
    (konsumsi_spesifik, (1, 0))
]:
    try:
        fungsi(*args)
        raise AssertionError(f"{fungsi.__name__} seharusnya menolak energi nol")
    except ValueError:
        pass

print("Semua uji assert lulus.")

Semua uji assert lulus.


In [ ]:
# Prosedur laporan

In [ ]:
def cetak_laporan(nama_unit, daya_mw, nilai_kalor, energi, batubara, jam_operasi):
    """Mencetak tabel KPI dan ringkasan untuk satu unit pembangkit."""
    if not (len(energi) == len(batubara) == len(jam_operasi)):
        raise ValueError("Panjang data energi, batubara, dan jam operasi harus sama")

    hasil = []
    print(f"\nLAPORAN KPI {nama_unit}")
    print(f"{'Hari':>4} {'HR (kcal/kWh)':>15} {'Efi (%)':>9} "
          f"{'CF (%)':>8} {'SCC (t/MWh)':>12}  Status")
    print("-" * 78)

    for i in range(len(energi)):
        hr = heat_rate(batubara[i], energi[i], nilai_kalor)
        efi = efisiensi_termal(hr)
        cf = capacity_factor(energi[i], daya_mw, jam_operasi[i])
        scc = konsumsi_spesifik(batubara[i], energi[i])
        status = status_hari(efi, jam_operasi[i])

        hasil.append({
            "hari": i + 1, "hr": hr, "efi": efi, "cf": cf,
            "scc": scc, "status": status
        })

        print(f"{i+1:>4} {hr:>15,.1f} {efi:>9.2f} {cf:>8.1f} "
              f"{scc:>12.3f}  {status}")

    total_jam = sum(jam_operasi)
    rata_efi = sum(item["efi"] for item in hasil) / len(hasil)
    cf_bulanan = capacity_factor(sum(energi), daya_mw, total_jam)

    print("\nRINGKASAN")
    print(f"Rata-rata efisiensi       : {rata_efi:.2f} %")
    print(f"Capacity factor periode  : {cf_bulanan:.2f} %")

    print("Jumlah hari per status:")
    for status in ["NORMAL", "EFISIENSI RENDAH", "TIDAK PENUH"]:
        jumlah = sum(1 for item in hasil if item["status"] == status)
        print(f"  {status:<18}: {jumlah} hari")

    print("Tiga hari paling tidak efisien:")
    tiga_terendah = sorted(hasil, key=lambda item: item["efi"])[:3]
    for item in tiga_terendah:
        print(f"  Hari {item['hari']:>2}: {item['efi']:.2f} % "
              f"({item['status']})")

# Uji prosedur cetak_laporan
from contextlib import redirect_stdout
from io import StringIO

hasil_uji = StringIO()
with redirect_stdout(hasil_uji):
    cetak_laporan("Uji", 330, 4400, [2], [1], [24])
assert "LAPORAN KPI Uji" in hasil_uji.getvalue()
assert "NORMAL" in hasil_uji.getvalue()

# Data Unit 1 – 1 sampai 30 November 2026 (disalin sesuai soal)
energi_unit1 = [
    5576, 6098, 6078, 6095, 6074, 6077, 5564, 5552, 6100, 6083,
    6097, 6089, 6066, 5533, 5566, 6097, 3592, 6059, 6100, 6098,
    5565, 5571, 6103, 6135, 6065, 6071, 6086, 5577, 5575, 6062
]
batubara_unit1 = [
    2998, 3258, 3250, 3263, 3249, 3246, 2993, 2983, 3258, 3258,
    3253, 3249, 3244, 2996, 3009, 3257, 1950, 3234, 3261, 3262,
    3015, 2995, 3260, 3275, 3248, 3264, 3267, 2998, 3008, 3240
]
jam_unit1 = [24] * 30
jam_unit1[16] = 18  # Hari ke-17 beroperasi 18 jam

cetak_laporan(
    "Unit 1", 330, 4400,
    energi_unit1, batubara_unit1, jam_unit1
)


LAPORAN KPI Unit 1
Hari   HR (kcal/kWh)   Efi (%)   CF (%)  SCC (t/MWh)  Status
------------------------------------------------------------------------------
   1         2,365.7     36.35     70.4        0.538  NORMAL
   2         2,350.8     36.58     77.0        0.534  NORMAL
   3         2,352.7     36.55     76.7        0.535  NORMAL
   4         2,355.6     36.51     77.0        0.535  NORMAL
   5         2,353.6     36.54     76.7        0.535  NORMAL
   6         2,350.2     36.59     76.7        0.534  NORMAL
   7         2,366.9     36.34     70.3        0.538  NORMAL
   8         2,364.0     36.38     70.1        0.537  NORMAL
   9         2,350.0     36.60     77.0        0.534  NORMAL
  10         2,356.6     36.49     76.8        0.536  NORMAL
  11         2,347.6     36.63     77.0        0.534  NORMAL
  12         2,347.8     36.63     76.9        0.534  NORMAL
  13         2,353.0     36.55     76.6        0.535  NORMAL
  14         2,382.5     36.10     69.9        

In [ ]:
# Penggunaan ulang fungsi untuk Unit 2

In [ ]:
energi_unit2 = [12150, 12080, 11020, 12210, 12095]
batubara_unit2 = [5240, 5225, 4790, 5260, 5230]
jam_unit2 = [24] * 5

cetak_laporan(
    "Unit 2", 600, 5100,
    energi_unit2, batubara_unit2, jam_unit2
)


LAPORAN KPI Unit 2
Hari   HR (kcal/kWh)   Efi (%)   CF (%)  SCC (t/MWh)  Status
------------------------------------------------------------------------------
   1         2,199.5     39.10     84.4        0.431  NORMAL
   2         2,205.9     38.99     83.9        0.433  NORMAL
   3         2,216.8     38.79     76.5        0.435  NORMAL
   4         2,197.1     39.14     84.8        0.431  NORMAL
   5         2,205.3     39.00     84.0        0.432  NORMAL

RINGKASAN
Rata-rata efisiensi       : 39.00 %
Capacity factor periode  : 82.72 %
Jumlah hari per status:
  NORMAL            : 5 hari
  EFISIENSI RENDAH  : 0 hari
  TIDAK PENUH       : 0 hari
Tiga hari paling tidak efisien:
  Hari  3: 38.79 % (NORMAL)
  Hari  2: 38.99 % (NORMAL)
  Hari  5: 39.00 % (NORMAL)
